# Clasificación Automática de Exoplanetas Kepler
Este notebook contiene el análisis de datos, preprocesamiento y entrenamiento del modelo de Machine Learning para clasificar Objetos de Interés de Kepler (KOI) en exoplanetas confirmados o falsos positivos.

### Paso 1: Importación de Librerías
Importamos las librerías necesarias para la manipulación de datos, división de conjuntos de datos, normalización, modelado, métricas y guardado del modelo.

In [1]:
# Importación de librerías base para análisis numérico y manipulación de datos
import pandas as pd
import numpy as np

# Importación de utilidades de scikit-learn para preprocesamiento y entrenamiento
from sklearn.model_selection import train_test_split  # Para dividir los datos en entrenamiento y prueba
from sklearn.preprocessing import StandardScaler      # Para normalizar las características continuas
from sklearn.ensemble import RandomForestClassifier     # Algoritmo de clasificación de ensamble
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score # Métricas de evaluación
import joblib # Para guardar y exportar el modelo y el escalador entrenados

ModuleNotFoundError: No module named 'pandas'

### Paso 2: Carga del Catálogo Acumulativo de Kepler
Cargamos el dataset oficial de avistamientos de Kepler en un DataFrame de pandas.

In [ ]:
# Cargar los datos acumulativos en formato CSV en un DataFrame de pandas
df = pd.read_csv("model/cumulative.csv")

### Paso 3: Exploración Inicial de los Datos
Inspeccionamos las primeras filas de la tabla para entender sus columnas y valores.

In [ ]:
# Mostrar las primeras 5 filas para una primera vista rápida de los datos
df.head()

NameError: name 'df' is not defined

### Paso 4: Limpieza y Filtrado de Clases Objetivo
Filtramos la variable objetivo (`koi_disposition`) para conservar solo las observaciones con confirmación definitiva: `CONFIRMED` y `FALSE POSITIVE`. Descartamos los registros marcados como `CANDIDATE` ya que aún no tienen una confirmación oficial y no nos servirían para el aprendizaje supervisado.

In [ ]:
# Filtrar el DataFrame original para conservar únicamente registros confirmados o falsos positivos conocidos
df = df[df['koi_disposition'].isin(['CONFIRMED', 'FALSE POSITIVE'])]

### Paso 5: Codificación Binaria de la Variable Objetivo
Mapeamos las etiquetas de texto a números binarios para que el modelo pueda procesarlos (1 para planetas confirmados, 0 para falsos positivos).


In [ ]:
# Crear la columna 'es_planeta' mapeando CONFIRMED a 1 y FALSE POSITIVE a 0
df['es_planeta'] = df['koi_disposition'].map({'CONFIRMED': 1, 'FALSE POSITIVE': 0})

### Paso 6: Selección de Características (Features)
Seleccionamos las características estelares, de coordenadas espaciales y banderas de falsos positivos más relevantes del catálogo. Cumplimos con el objetivo de utilizar más de 6 columnas de datos.

In [ ]:
# Lista de variables estelares y banderas predictivas que se ingresarán al modelo
columnas_importantes = [
    'koi_fpflag_nt',  # Anomalía: Bandera de curva de luz no compatible con tránsito planetario regular
    'koi_fpflag_ss',  # Anomalía: Bandera de eclipse binario o co-tránsito estelar de fondo detectado
    'koi_fpflag_co',  # Anomalía: Bandera de desplazamiento del centroide lumínico durante el tránsito
    'koi_slogg',      # Característica física: Gravedad en la superficie de la estrella (log(g))
    'koi_srad',       # Característica física: Radio estelar medido en múltiplos del radio solar
    'ra',             # Coordenada celeste: Ascensión Recta (longitud celeste)
    'dec',            # Coordenada celeste: Declinación (latitud celeste)
    'koi_kepmag',     # Brillo estelar aparente medido por el telescopio espacial Kepler
    'es_planeta'      # Variable objetivo (Clasificación real del catálogo)
]

### Paso 7: Filtrado y Tratamiento de Valores Nulos
Creamos un subconjunto con las columnas seleccionadas y removemos filas con valores faltantes para evitar errores en las operaciones de álgebra lineal y optimización del clasificador.

In [ ]:
# Copiar el dataframe con las columnas seleccionadas
df_filtrado = df[columnas_importantes].copy()

# Eliminar registros con valores faltantes (NaN) en cualquiera de las columnas seleccionadas
df_filtrado = df_filtrado.dropna()

### Paso 8: Inspección del Dataset Limpio
Imprimimos las dimensiones del conjunto final antes de iniciar el análisis exploratorio y la separación.

In [ ]:
# Mostrar el número final de registros y columnas que serán utilizados para entrenar el modelo
print(f"Tamaño del dataset listo: {df_filtrado.shape}")


### Paso 9: Análisis Exploratorio de Datos (EDA)
Graficamos la correlación de las variables seleccionadas y evaluamos si el dataset tiene balance o desbalance en sus clases de clasificación.

In [ ]:
# Importación de librerías para gráficos visuales
import matplotlib.pyplot as plt
import seaborn as sns

# Graficar matriz de correlación de Pearson para identificar dependencias lineales entre variables
plt.figure(figsize=(10, 8))
sns.heatmap(df_filtrado.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Matriz de Correlación de Variables Estelares y Banderas")
plt.show()

# Graficar la distribución de clases reales (Exoplanetas Confirmados vs Falsos Positivos)
sns.countplot(x='es_planeta', data=df_filtrado, hue='es_planeta', palette='viridis', legend=False)
plt.title("Distribución de Clases (1 = Planeta, 0 = Falso Positivo)")
plt.show()

### Paso 10: Separación y Escalamiento de Datos
Dividimos las variables en características (X) y objetivo (y). Luego dividimos el set en un 80% para entrenamiento (`train`) y 20% para evaluación (`test`), aplicando estratificación para preservar el balance de clases. Finalmente, estandarizamos las variables utilizando `StandardScaler` para que todas tengan media 0 y varianza 1.

In [ ]:
# Separar el dataframe en variables predictoras (X) y variable objetivo (y)
X = df_filtrado.drop('es_planeta', axis=1)
y = df_filtrado['es_planeta']

# Dividir el dataset de manera aleatoria en 80% entrenamiento y 20% prueba
# El parámetro stratify asegura que la proporción de clases se mantenga idéntica en ambos sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Instanciar y ajustar el escalador estándar con los datos de entrenamiento
# Luego transformamos tanto el set de entrenamiento como el de pruebas
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)